# 🔍 Capa 2: Funciones de Consulta

Este notebook define las funciones que el chatbot usará para consultar la base de datos.

**Funciones:**
- `buscar_producto(nombre)` — Busca productos por nombre
- `buscar_por_categoria(categoria)` — Lista productos de una categoría
- `consultar_stock(producto_id)` — Verifica disponibilidad
- `consultar_pedidos(email)` — Estado de pedidos por email
- `obtener_detalle_pedido(pedido_id)` — Detalle de un pedido específico
- `obtener_promociones()` — Promociones vigentes
- `obtener_promocion_producto(producto_id)` — Promo de un producto
- `registrar_devolucion(...)` — Crea solicitud de devolución
- `consultar_devolucion(email)` — Estado de devoluciones
- `crear_ticket_soporte(...)` — Escala a agente humano

In [25]:
import sqlite3
import os
import unicodedata
from datetime import datetime
import pandas as pd

DB_PATH = '../data/ecomarket.db'

def normalizar_texto(texto: str) -> str:
    """Quita acentos y pasa a minúsculas para búsquedas flexibles."""
    texto = unicodedata.normalize('NFD', (texto or '').lower())
    return ''.join(c for c in texto if unicodedata.category(c) != 'Mn')

def get_connection():
    """Obtiene conexión a la base de datos."""
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row  # Para acceder por nombre de columna
    return conn

## 1. Consultas de Productos

In [26]:
def buscar_producto(nombre: str) -> list[dict]:
    """Busca productos por nombre (parcial, ignora mayúsculas y acentos)."""
    conn = get_connection()
    cursor = conn.cursor()
    termino = normalizar_texto(nombre)
    cursor.execute('''
        SELECT id, nombre, categoria, precio, stock, descripcion, unidad
        FROM productos
    ''')
    resultados = [
        dict(row) for row in cursor.fetchall()
        if termino in normalizar_texto(dict(row)['nombre'])
    ]
    conn.close()
    return resultados


def buscar_por_categoria(categoria: str) -> list[dict]:
    """Lista todos los productos de una categoría."""
    conn = get_connection()
    cursor = conn.cursor()
    termino = normalizar_texto(categoria)
    cursor.execute('''
        SELECT id, nombre, categoria, precio, stock, unidad
        FROM productos
    ''')
    resultados = [
        dict(row) for row in cursor.fetchall()
        if termino in normalizar_texto(dict(row)['categoria'])
    ]
    resultados.sort(key=lambda p: p['nombre'])
    conn.close()
    return resultados


def consultar_stock(producto_id: int) -> dict:
    """Verifica stock de un producto por ID."""
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute('''
        SELECT nombre, stock, unidad
        FROM productos WHERE id = ?
    ''', (producto_id,))
    
    row = cursor.fetchone()
    conn.close()
    
    if row:
        return dict(row)
    return {'error': 'Producto no encontrado'}


def obtener_categorias() -> list[str]:
    """Devuelve las categorías disponibles."""
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute('SELECT DISTINCT categoria FROM productos ORDER BY categoria')
    categorias = [row[0] for row in cursor.fetchall()]
    conn.close()
    return categorias

## 2. Consultas de Pedidos

In [27]:
def consultar_pedidos(email: str) -> list[dict]:
    """Obtiene todos los pedidos de un cliente por email."""
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute('''
        SELECT id, fecha_pedido, estado, total, direccion_envio, fecha_entrega_estimada
        FROM pedidos
        WHERE LOWER(email_cliente) = LOWER(?)
        ORDER BY fecha_pedido DESC
    ''', (email,))
    
    resultados = [dict(row) for row in cursor.fetchall()]
    conn.close()
    return resultados


def obtener_detalle_pedido(pedido_id: int) -> dict:
    """Obtiene el detalle completo de un pedido."""
    conn = get_connection()
    cursor = conn.cursor()
    
    # Info general del pedido
    cursor.execute('SELECT * FROM pedidos WHERE id = ?', (pedido_id,))
    pedido = cursor.fetchone()
    
    if not pedido:
        conn.close()
        return {'error': 'Pedido no encontrado'}
    
    # Items del pedido
    cursor.execute('''
        SELECT p.nombre, dp.cantidad, dp.precio_unitario,
               (dp.cantidad * dp.precio_unitario) as subtotal
        FROM detalle_pedido dp
        JOIN productos p ON p.id = dp.producto_id
        WHERE dp.pedido_id = ?
    ''', (pedido_id,))
    
    items = [dict(row) for row in cursor.fetchall()]
    conn.close()
    
    return {
        'pedido': dict(pedido),
        'items': items
    }

## 3. Consultas de Promociones

In [28]:
def obtener_promociones() -> list[dict]:
    """Obtiene todas las promociones vigentes."""
    conn = get_connection()
    cursor = conn.cursor()
    hoy = datetime.now().strftime('%Y-%m-%d')
    
    cursor.execute('''
        SELECT pr.id, p.nombre as producto, pr.descripcion,
               pr.descuento_porcentaje, pr.fecha_inicio, pr.fecha_fin,
               p.precio as precio_original,
               ROUND(p.precio * (1 - pr.descuento_porcentaje/100), 2) as precio_con_descuento
        FROM promociones pr
        JOIN productos p ON p.id = pr.producto_id
        WHERE pr.activa = 1
          AND pr.fecha_inicio <= ?
          AND pr.fecha_fin >= ?
        ORDER BY pr.descuento_porcentaje DESC
    ''', (hoy, hoy))
    
    resultados = [dict(row) for row in cursor.fetchall()]
    conn.close()
    return resultados


def obtener_promocion_producto(producto_id: int) -> dict | None:
    """Busca si un producto tiene promoción activa."""
    conn = get_connection()
    cursor = conn.cursor()
    hoy = datetime.now().strftime('%Y-%m-%d')
    
    cursor.execute('''
        SELECT pr.descripcion, pr.descuento_porcentaje, pr.fecha_fin,
               p.precio as precio_original,
               ROUND(p.precio * (1 - pr.descuento_porcentaje/100), 2) as precio_con_descuento
        FROM promociones pr
        JOIN productos p ON p.id = pr.producto_id
        WHERE pr.producto_id = ?
          AND pr.activa = 1
          AND pr.fecha_inicio <= ?
          AND pr.fecha_fin >= ?
    ''', (producto_id, hoy, hoy))
    
    row = cursor.fetchone()
    conn.close()
    
    if row:
        return dict(row)
    return None

## 4. Gestión de Devoluciones

In [29]:
def registrar_devolucion(pedido_id: int, email: str, motivo: str) -> dict:
    """Registra una solicitud de devolución."""
    conn = get_connection()
    cursor = conn.cursor()
    
    # Verificar que el pedido existe y pertenece al cliente
    cursor.execute('''
        SELECT id, estado FROM pedidos
        WHERE id = ? AND LOWER(email_cliente) = LOWER(?)
    ''', (pedido_id, email))
    
    pedido = cursor.fetchone()
    if not pedido:
        conn.close()
        return {'error': 'Pedido no encontrado o no pertenece a este email'}
    
    if dict(pedido)['estado'] not in ['entregado', 'enviado']:
        conn.close()
        return {'error': f'No se puede devolver un pedido con estado: {dict(pedido)["estado"]}'}
    
    # Registrar devolución
    fecha = datetime.now().strftime('%Y-%m-%d')
    cursor.execute('''
        INSERT INTO devoluciones (pedido_id, email_cliente, motivo, estado, fecha_solicitud)
        VALUES (?, ?, ?, 'pendiente', ?)
    ''', (pedido_id, email, motivo, fecha))
    
    devolucion_id = cursor.lastrowid
    conn.commit()
    conn.close()
    
    return {
        'mensaje': 'Devolución registrada correctamente',
        'devolucion_id': devolucion_id,
        'estado': 'pendiente'
    }


def consultar_devoluciones(email: str) -> list[dict]:
    """Consulta las devoluciones de un cliente."""
    conn = get_connection()
    cursor = conn.cursor()
    
    cursor.execute('''
        SELECT d.id, d.pedido_id, d.motivo, d.estado,
               d.fecha_solicitud, d.fecha_resolucion, d.comentarios
        FROM devoluciones d
        WHERE LOWER(d.email_cliente) = LOWER(?)
        ORDER BY d.fecha_solicitud DESC
    ''', (email,))
    
    resultados = [dict(row) for row in cursor.fetchall()]
    conn.close()
    return resultados

## 5. Escalado a Humano (Tickets de Soporte)

In [30]:
def crear_ticket_soporte(email: str, asunto: str, descripcion: str, pedido_id: int = None) -> dict:
    """Crea un ticket de soporte para derivar a agente humano."""
    conn = get_connection()
    cursor = conn.cursor()
    
    fecha = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    cursor.execute('''
        INSERT INTO tickets_soporte (email_cliente, asunto, descripcion, estado, fecha_creacion, pedido_id)
        VALUES (?, ?, ?, 'abierto', ?, ?)
    ''', (email, asunto, descripcion, fecha, pedido_id))
    
    ticket_id = cursor.lastrowid
    conn.commit()
    conn.close()
    
    return {
        'mensaje': 'Ticket creado. Un agente se pondrá en contacto contigo pronto.',
        'ticket_id': ticket_id,
        'estado': 'abierto'
    }

## 6. Pruebas rápidas

In [31]:
# Probar búsqueda de productos
print('🔍 Buscar "leche":')
for p in buscar_producto('leche'):
    print(f"  - {p['nombre']} ({p['categoria']}) - €{p['precio']} - Stock: {p['stock']}")

print('\n📂 Categorías disponibles:')
for cat in obtener_categorias():
    print(f'  - {cat}')

print('\n🥩 Productos en Carnes:')
for p in buscar_por_categoria('carnes'):
    print(f"  - {p['nombre']} - €{p['precio']}")

🔍 Buscar "leche":
  - Leche Entera (Lacteos) - €1.2 - Stock: 300
  - Leche de Almendra (Lacteos) - €2.8 - Stock: 70

📂 Categorías disponibles:
  - Bebidas
  - Carnes
  - Despensa
  - Frutas y Verduras
  - Lacteos
  - Limpieza y Hogar
  - Panaderia

🥩 Productos en Carnes:
  - Carne Molida - €8.5
  - Chorizo - €4.5
  - Jamon Serrano - €15.0
  - Pechuga de Pollo - €6.9
  - Salmon Fresco - €12.0


In [32]:
# Probar consulta de pedidos
print('📦 Pedidos de maria.garcia@email.com:')
pedidos = consultar_pedidos('maria.garcia@email.com')
for p in pedidos:
    print(f"  Pedido #{p['id']} - {p['estado']} - €{p['total']} - {p['fecha_pedido']}")

if pedidos:
    print(f"\n📋 Detalle del pedido #{pedidos[0]['id']}:")
    detalle = obtener_detalle_pedido(pedidos[0]['id'])
    for item in detalle['items']:
        print(f"  - {item['nombre']} x{item['cantidad']} = €{item['subtotal']:.2f}")

📦 Pedidos de maria.garcia@email.com:
  Pedido #15 - enviado - €57.12 - 2026-05-24

📋 Detalle del pedido #15:
  - Mantequilla x1 = €2.30
  - Te Verde x2 = €6.40


In [33]:
# Probar promociones
print('🏷️ Promociones vigentes:')
for promo in obtener_promociones():
    print(f"  - {promo['producto']}: {promo['descripcion']}")
    print(f"    Precio: €{promo['precio_original']} → €{promo['precio_con_descuento']} (-{promo['descuento_porcentaje']}%)")

🏷️ Promociones vigentes:
  - Manzana Roja: 2x1 en Manzanas Rojas
    Precio: €2.5 → €1.25 (-50.0%)
  - Pan Integral: Pan Integral a mitad de precio los martes
    Precio: €2.1 → €1.05 (-50.0%)
  - Agua Mineral: Agua Mineral: 3x2
    Precio: €0.7 → €0.47 (-33.0%)
  - Chocolate Negro: 25% descuento en Chocolate Negro
    Precio: €3.5 → €2.63 (-25.0%)
  - Cerveza Artesanal: Cerveza Artesanal: 4x3
    Precio: €3.5 → €2.63 (-25.0%)
  - Leche Entera: 20% descuento en Leche Entera
    Precio: €1.2 → €0.96 (-20.0%)
  - Pechuga de Pollo: 15% descuento en Pechuga de Pollo
    Precio: €6.9 → €5.87 (-15.0%)
  - Aceite de Oliva: 10% descuento en Aceite de Oliva
    Precio: €7.0 → €6.3 (-10.0%)


In [34]:
# Probar devoluciones
print('🔄 Devoluciones de ana.martinez@email.com:')
for d in consultar_devoluciones('ana.martinez@email.com'):
    print(f"  Devolución #{d['id']} - Pedido #{d['pedido_id']} - Estado: {d['estado']}")
    print(f"  Motivo: {d['motivo']}")

🔄 Devoluciones de ana.martinez@email.com:
  Devolución #1 - Pedido #3 - Estado: aprobada
  Motivo: Producto llego danado


## 6. Crear Pedidos

Permite a los clientes hacer pedidos a través del chatbot.
El pedido se inserta en las tablas `pedidos` y `detalle_pedido`, y se actualiza el stock.

In [35]:
from datetime import timedelta

def crear_pedido(email: str, productos_cantidades: list, direccion: str) -> dict:
    """
    Crea un nuevo pedido y actualiza el stock.
    
    Args:
        email: Email del cliente
        productos_cantidades: Lista de dicts con 'nombre' y 'cantidad'
            Ej: [{'nombre': 'Leche Entera', 'cantidad': 2}]
        direccion: Dirección de entrega
    
    Returns:
        Dict con confirmación del pedido o error
    """
    conn = get_connection()
    cursor = conn.cursor()
    
    items_pedido = []
    total = 0.0
    errores = []
    
    # Verificar cada producto
    for item in productos_cantidades:
        nombre_prod = item.get('nombre', '')
        cantidad = item.get('cantidad', 1)
        
        termino = normalizar_texto(nombre_prod)
        cursor.execute('SELECT id, nombre, precio, stock FROM productos')
        coincidencias = [
            dict(row) for row in cursor.fetchall()
            if termino in normalizar_texto(dict(row)['nombre'])
        ]
        producto = coincidencias[0] if coincidencias else None
        
        if not producto:
            errores.append(f'Producto "{nombre_prod}" no encontrado')
            continue
        
        prod_dict = dict(producto)
        
        if prod_dict['stock'] < cantidad:
            errores.append(f'{prod_dict["nombre"]}: stock insuficiente (disponible: {prod_dict["stock"]})')
            continue
        
        items_pedido.append({
            'producto_id': prod_dict['id'],
            'nombre': prod_dict['nombre'],
            'cantidad': cantidad,
            'precio_unitario': prod_dict['precio'],
            'subtotal': round(prod_dict['precio'] * cantidad, 2)
        })
        total += prod_dict['precio'] * cantidad
    
    if errores:
        conn.close()
        return {'error': 'No se pudo crear el pedido', 'problemas': errores}
    
    if not items_pedido:
        conn.close()
        return {'error': 'No se encontraron productos válidos para el pedido'}
    
    # Crear el pedido
    fecha = datetime.now().strftime('%Y-%m-%d')
    fecha_entrega = (datetime.now() + timedelta(days=2)).strftime('%Y-%m-%d')
    total = round(total, 2)
    
    cursor.execute(
        'INSERT INTO pedidos (email_cliente, fecha_pedido, estado, total, direccion_envio, fecha_entrega_estimada) '
        'VALUES (?, ?, "confirmado", ?, ?, ?)',
        (email, fecha, total, direccion, fecha_entrega)
    )
    pedido_id = cursor.lastrowid
    
    # Insertar detalle y actualizar stock
    for item in items_pedido:
        cursor.execute(
            'INSERT INTO detalle_pedido (pedido_id, producto_id, cantidad, precio_unitario) '
            'VALUES (?, ?, ?, ?)',
            (pedido_id, item['producto_id'], item['cantidad'], item['precio_unitario'])
        )
        cursor.execute(
            'UPDATE productos SET stock = stock - ? WHERE id = ?',
            (item['cantidad'], item['producto_id'])
        )
    
    conn.commit()
    conn.close()
    
    return {
        'mensaje': 'Pedido creado exitosamente',
        'pedido_id': pedido_id,
        'total': total,
        'estado': 'confirmado',
        'metodo_pago': 'Pago contra entrega (se paga al domiciliario)',
        'fecha_entrega_estimada': fecha_entrega,
        'direccion': direccion,
        'items': [{'nombre': i['nombre'], 'cantidad': i['cantidad'], 'subtotal': i['subtotal']} for i in items_pedido]
    }

print('Función crear_pedido definida ✅')

Función crear_pedido definida ✅


In [36]:
# Probar crear un pedido
print('🛒 Creando pedido de prueba...')
resultado = crear_pedido(
    email='maria.garcia@email.com',
    productos_cantidades=[
        {'nombre': 'Leche Entera', 'cantidad': 2},
        {'nombre': 'Pan Integral', 'cantidad': 1},
        {'nombre': 'Aguacate', 'cantidad': 3}
    ],
    direccion='Calle Mayor 15, Madrid'
)

print(f"Resultado: {resultado}")

# Verificar que aparece en los pedidos del cliente
print('\n📦 Pedidos actualizados de maria.garcia@email.com:')
for p in consultar_pedidos('maria.garcia@email.com'):
    print(f"  Pedido #{p['id']} - {p['estado']} - €{p['total']} - {p['fecha_pedido']}")

🛒 Creando pedido de prueba...
Resultado: {'mensaje': 'Pedido creado exitosamente', 'pedido_id': 21, 'total': 18.0, 'estado': 'confirmado', 'metodo_pago': 'Pago contra entrega (se paga al domiciliario)', 'fecha_entrega_estimada': '2026-06-20', 'direccion': 'Calle Mayor 15, Madrid', 'items': [{'nombre': 'Leche Entera', 'cantidad': 2, 'subtotal': 2.4}, {'nombre': 'Pan Integral', 'cantidad': 1, 'subtotal': 2.1}, {'nombre': 'Aguacate', 'cantidad': 3, 'subtotal': 13.5}]}

📦 Pedidos actualizados de maria.garcia@email.com:
  Pedido #21 - confirmado - €18.0 - 2026-06-18
  Pedido #15 - enviado - €57.12 - 2026-05-24
